# Grad-CAM — Expertos ISIC (EfficientNet-B3) y Páncreas (R3D-18)

**Datos:** descompresión y rutas alineadas con `notebooks/05_Router_Vit_Lineal_Solo.ipynb` (`RAW_DIR`, `LOCAL_DEST`, `DATASET_ROOTS`).

Los notebooks de entrenamiento `ISIC2019_EfficientNetB3_Training_Final.ipynb` y `Pancreas_R3D18_Training.ipynb` no incluyen visualización Grad-CAM. Este notebook **carga los pesos entrenados**, **descomprime solo los datos necesarios** y aplica la **misma metodología XAI** que `Osteoarthritis_VGG16BN_Training.ipynb` (**Grad-CAM**, **Score-CAM**, **Layer-CAM** vía `pytorch_grad_cam`) para ISIC; para el volumen 3D del páncreas se usa **Grad-CAM 3D manual** sobre la última capa convolucional espacial (`layer4` de R3D-18), proyectando cortes axiales.

**Inferencia en CPU (Colab):** Sí. Puedes usar un runtime **sin GPU**: `torch` ejecuta el forward/backward en CPU; Grad-CAM funciona igual. Será **más lento** (sobre todo Score-CAM y el volumen 3D). Para ISIC suele ser aceptable; para Páncreas reduce `NUM_PAN_SAMPLES` o prueba solo cortes intermedios.

## 0. Instalación

In [ ]:
!pip install -q grad-cam timm monai torch torchvision pillow opencv-python pandas matplotlib numpy scikit-learn SimpleITK


## 1. Rutas, Drive y candidatos de pesos (MoE)

In [ ]:
import os
import glob
import shutil
import subprocess
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.video import r3d_18, R3D_18_Weights
import timm

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# --- Rutas: mismas cadenas que notebooks/05_Router_Vit_Lineal_Solo.ipynb ---
RAW_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/"
LOCAL_DEST = "/content/datasets/"
WEIGHTS_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/"
os.makedirs(LOCAL_DEST, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

DATASET_ROOTS = {
    "NIH":      (os.path.join(LOCAL_DEST, "NIH Chest X ray 14"), 0),
    "ISIC":     (os.path.join(LOCAL_DEST, "ISIC 2019"), 1),
    "Osteo":    (os.path.join(LOCAL_DEST, "Knee Osteoarthritis Classification"), 2),
    "LUNA16":   (os.path.join(LOCAL_DEST, "Luna16 Lung Cancer Dataset"), 3),
    "Pancreas": (os.path.join(LOCAL_DEST, "Pancreas Cancer"), 4),
}

print("Rutas (datasets tras extracción en LOCAL_DEST):")
for k, (p, d) in DATASET_ROOTS.items():
    ok = os.path.isdir(p)
    print(f"  {k} id={d} exists={ok} -> {p}")
print(f"RAW_DIR={RAW_DIR}")
print(f"LOCAL_DEST={LOCAL_DEST}")
print(f"WEIGHTS_DIR={WEIGHTS_DIR}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


## 1b. Arquitecturas expertos y checkpoints (igual que `05_Router_Vit_Lineal_Solo`)

`build_expert`, `_default_checkpoint_candidates`, `resolve_checkpoint` (incluye **`.pth.zip`**), `load_weights`.

In [ ]:
# =============================================================================
# Arquitecturas de los 5 expertos (embebido en el notebook; sin .py externo)
# =============================================================================

import os
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import timm
import torchvision.models as models
from torchvision.models.video import R3D_18_Weights, r3d_18


# ============================================================================
# Experto 1 (NIH) - Swin-Tiny (NIH_ChestXray_Swin_Tiny_Training.ipynb)
# 5 clases multietiqueta (Mass, Nodule, Effusion, Cardiomegaly, Pneumothorax).
# Pesos: suele guardarse como Experts_2D/MaxViT_NIH_5cls.pth (nombre historico en CONFIG).
# ============================================================================
class SwinNIHClassifier(nn.Module):
    """Misma envoltura que SwinClassifier en el notebook de entrenamiento (timm)."""

    def __init__(self, num_classes: int = 5, pretrained: bool = True):
        super().__init__()
        self.model = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


# ============================================================================
# Experto 3 (Osteo) - VGG16-BN
# ============================================================================
def build_vgg16_bn(num_classes: int = 5, pretrained: bool = True):
    model = models.vgg16_bn(weights="IMAGENET1K_V1" if pretrained else None)
    old_conv = model.features[0]
    new_conv = nn.Conv2d(1, 64, kernel_size=3, padding=1)
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        new_conv.bias.copy_(old_conv.bias)
    model.features[0] = new_conv
    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 512),
        nn.ReLU(True),
        nn.BatchNorm1d(512),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(True),
        nn.BatchNorm1d(256),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model


# ============================================================================
# Experto 4 (LUNA16 3D) — R3D-18 Kinetics 3ch (LUNA16_R3D18_Training.ipynb)
# Experto 5 (Pancreas 3D) — R3D18 entrada 1 canal (stem adaptado)
# ============================================================================
from torch.utils.checkpoint import checkpoint as grad_checkpoint


class R3D18LunaKineticsExpert(nn.Module):
    """R3D-18 torchvision binario: 3 canales con stats Kinetics (ver luna_1ch_to_kinetics_3ch)."""

    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        base = r3d_18(weights=weights)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.head(feat)


class R3D18Expert(nn.Module):
    """R3D-18 para 3D binario (Exp5 Pancreas): entrada 1 canal."""

    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        base = r3d_18(weights=weights)
        old_conv = base.stem[0]
        stem_conv = nn.Conv3d(
            1,
            64,
            kernel_size=(3, 7, 7),
            stride=(1, 2, 2),
            padding=(1, 3, 3),
            bias=False,
        )
        with torch.no_grad():
            stem_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        base.stem[0] = stem_conv
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        # Gradient checkpointing obligatorio para expertos 3D (consigna §8.1)
        for block in self.backbone:
            x = grad_checkpoint(block, x, use_reentrant=False)
        return self.head(x.flatten(1))


# ============================================================================
# Factory y carga de pesos
# ============================================================================
import glob
import subprocess

EXPERT_SPECS = {
    1: {"name": "exp1_nih", "num_classes": 5, "arch": "swin_tiny_nih"},
    2: {"name": "exp2_isic", "num_classes": 9, "arch": "efficientnet_b3"},
    3: {"name": "exp3_osteo", "num_classes": 5, "arch": "vgg16_bn"},
    4: {
        "name": "exp4_luna16",
        "model_name": "R3D-18 (Kinetics 3ch)",
        "num_classes": 2,
        "arch": "r3d18_luna",
    },
    5: {"name": "exp5_pancreas", "num_classes": 2, "arch": "r3d18"},
}


def build_expert(expert_id: int, pretrained_backbone: bool = True):
    spec = EXPERT_SPECS[int(expert_id)]
    arch = spec["arch"]
    num_classes = spec["num_classes"]

    if arch == "swin_tiny_nih":
        return SwinNIHClassifier(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "efficientnet_b3":
        return timm.create_model("efficientnet_b3", pretrained=pretrained_backbone, num_classes=num_classes)
    if arch == "vgg16_bn":
        return build_vgg16_bn(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "r3d18_luna":
        return R3D18LunaKineticsExpert(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "r3d18":
        return R3D18Expert(num_classes=num_classes, pretrained=pretrained_backbone)
    raise ValueError(f"Arquitectura no soportada: {arch}")


def _default_checkpoint_candidates(weights_dir: str) -> Dict[int, List[str]]:
    """
    Candidatos por experto para tolerar nombres antiguos/nuevos de checkpoints.
    """
    return {
        1: [
            os.path.join(weights_dir, "Experts_2D", "MaxViT_NIH_5cls.pth"),
            os.path.join(weights_dir, "MaxViT_NIH_5cls.pth"),
            os.path.join(weights_dir, "exp1_NIH_SwinTiny_best.pth"),
            os.path.join(weights_dir, "exp1_NIH_LungMaxViT_best.pth"),
        ],
        2: [
            os.path.join(weights_dir, "exp2_ISIC_EfficientNetB3_best.pth"),
        ],
        3: [
            os.path.join(weights_dir, "exp3_Osteo_VGG16BN_best.pth"),
        ],
        4: [
            # Mismas rutas que antes; pesos del entrenamiento LUNA R3D-18 (expert4_*_best.pth)
            os.path.join(weights_dir, "exp4_LUNA16_3D_best.pth"),
        ],
        5: [
            os.path.join(weights_dir, "exp5_Pancreas_3D_best.pth"),
            os.path.join(weights_dir, "r3d18_pancreas_best_V2.pth"),
        ],
    }


def resolve_checkpoint(candidates: List[str]) -> str:
    """
    Busca el checkpoint. Si existe ``<ruta>.pth.zip`` y no el ``.pth``,
    descomprime en el mismo directorio y devuelve la ruta del ``.pth``.
    """
    for p in candidates:
        if any(ch in p for ch in ["*", "?", "["]):
            matches = sorted(glob.glob(p))
            if matches:
                return matches[0]
        if os.path.exists(p):
            return p
        zip_path = p + ".zip"
        if os.path.exists(zip_path):
            print(f"📦 Detectado peso comprimido: {os.path.basename(zip_path)}")
            extract_dir = os.path.dirname(zip_path) or "."
            try:
                print(f"  → Descomprimiendo en {extract_dir}...")
                subprocess.run(["unzip", "-o", zip_path, "-d", extract_dir], check=True)
                if os.path.exists(p):
                    print(f"  ✅ Checkpoint extraído: {os.path.basename(p)}")
                    return p
                new_matches = glob.glob(os.path.join(extract_dir, "*.pth"))
                if new_matches:
                    return sorted(new_matches)[0]
            except Exception as e:
                print(f"  ❌ Error al descomprimir {zip_path}: {e}")
    return ""


def load_weights(model: nn.Module, ckpt_path: str, map_location: str = "cpu", strict: bool = False):
    if not ckpt_path or not os.path.exists(ckpt_path):
        return False, "checkpoint no encontrado"
    raw = torch.load(ckpt_path, map_location=map_location)

    # Checkpoints de entrenamiento suelen ser dict con 'state_dict'
    if isinstance(raw, dict):
        if "state_dict" in raw:
            state = raw["state_dict"]
        elif "model_state_dict" in raw:
            state = raw["model_state_dict"]
        elif "model" in raw and isinstance(raw["model"], dict):
            state = raw["model"]
        else:
            state = raw
    else:
        state = raw

    try:
        model.load_state_dict(state, strict=strict)
        return True, "ok"
    except Exception as e:
        return False, f"load_state_dict fallo: {e}"

def freeze_and_eval(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model


def load_all_experts_from_drive(
    weights_dir: str,
    device: str = "cpu",
    strict: bool = False,
    pretrained_backbone: bool = False,
) -> Tuple[Dict[int, nn.Module], Dict[int, dict]]:
    """
    Crea y carga los 5 expertos con pesos desde Drive.

    Retorna:
      experts: dict[int, nn.Module]
      info: dict[int, {arch, ckpt, loaded, params}]
    """
    candidates = _default_checkpoint_candidates(weights_dir)
    experts = {}
    info = {}

    for eid in sorted(EXPERT_SPECS.keys()):
        model = build_expert(eid, pretrained_backbone=pretrained_backbone)
        ckpt_path = resolve_checkpoint(candidates[eid])
        loaded, msg = load_weights(model, ckpt_path, map_location="cpu", strict=strict)

        model = freeze_and_eval(model).to(device)
        experts[eid] = model
        spec = EXPERT_SPECS[eid]
        info[eid] = {
            "name": spec["name"],
            "arch": spec["arch"],
            "model_name": spec.get("model_name", ""),
            "ckpt": ckpt_path,
            "loaded": loaded,
            "message": msg,
            "params": int(sum(p.numel() for p in model.parameters())),
        }

    return experts, info


def print_expert_load_report(info: Dict[int, dict]):
    print("=== Expert Load Report ===")
    for eid in sorted(info.keys()):
        row = info[eid]
        status = "OK" if row["loaded"] else "MISSING"
        mn = row.get("model_name") or ""
        mn_s = f" | {mn}" if mn else ""
        print(
            f"Exp{eid} | {row['name']} | arch={row['arch']}{mn_s} | {status} | "
            f"params={row['params']:,} | ckpt={row['ckpt'] or 'N/A'}"
        )
print("Arquitecturas de expertos: definidas en el notebook.")


## 2. Descompresión (misma lógica que `05_Router_Vit_Lineal_Solo`)

`extract_datasets_colab` copia cada `.zip` desde `RAW_DIR` a `LOCAL_DEST`, descomprime en una carpeta con el **mismo nombre base** que el zip (p. ej. `ISIC 2019.zip` → `.../datasets/ISIC 2019/`) y descomprime **ZIPs internos** recursivamente — igual que el router.

- **`EXTRACT_ZIP_STEMS_ONLY`**: si es una tupla de nombres **sin `.zip`** (p. ej. `("ISIC 2019", "Pancreas Cancer")`), solo procesa esos archivos en `RAW_DIR`. Si es `None`, procesa **todos** los zip (comportamiento del 05).
- **`RUN_EXTRACT_ZIPS`**: pon `False` si ya tienes las carpetas bajo `/content/datasets/`.
- **Páncreas `.npz`**: el zip `Pancreas Cancer` trae NIfTI/MHA; los **volúmenes NPZ 64³** salen del EDA (`Pancreas_eda_*`) y suelen estar en **`/content/dataset_npz`** o en **`dataset_npz_procesado.zip`** en la raíz del proyecto. La celda siguiente intenta descomprimir ese zip a `/content/dataset_npz`.

Si sigue sin encontrar zip: comprueba en Drive que los archivos estén en `PROYECTO_MOE_VISION/01_Data/Raw/` y que el nombre base del zip coincida (espacios incluidos).

In [ ]:
def extract_datasets_colab(raw_dir=RAW_DIR, local_dest=LOCAL_DEST, only_zip_stems=None):
    """Copia ZIPs de Drive a LOCAL_DEST y los descomprime (misma función que 05)."""
    if not os.path.exists(raw_dir):
        print(f"Ruta {raw_dir} no existe. No se extrae nada.")
        return
    zip_files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(".zip")])
    if only_zip_stems is not None:
        allow = set(only_zip_stems)
        zip_files = [f for f in zip_files if os.path.splitext(f)[0] in allow]
        print(f"Filtrado por nombre de dataset: {len(zip_files)} zip(s) -> {zip_files}")
    print(f"Encontrados {len(zip_files)} zip(s) a procesar.")
    for zip_name in zip_files:
        print("=" * 60)
        print(f"Procesando: {zip_name}")
        drive_zip_path = os.path.join(raw_dir, zip_name)
        dataset_name = os.path.splitext(zip_name)[0]
        unzip_dir = os.path.join(local_dest, dataset_name)
        local_zip_path = os.path.join(local_dest, zip_name)
        if os.path.isdir(unzip_dir) and len(os.listdir(unzip_dir)) > 0:
            print(f" Ya existe: {unzip_dir} (omitido).")
            continue
        print(" 1. Copiando ZIP...")
        shutil.copy2(drive_zip_path, local_zip_path)
        os.makedirs(unzip_dir, exist_ok=True)
        print(f" 2. Descomprimiendo en {unzip_dir}...")
        subprocess.run(["unzip", "-q", local_zip_path, "-d", unzip_dir], check=True)
        print(" 3. Borrando ZIP local.")
        os.remove(local_zip_path)
        for iz in glob.glob(os.path.join(unzip_dir, "**", "*.zip"), recursive=True):
            print(f" -> ZIP interno: {iz}")
            subprocess.run(["unzip", "-q", iz, "-d", os.path.dirname(iz)], check=True)
            os.remove(iz)
    print("\nExtracción completa.")


# --- NPZ páncreas (EDA / R3D): NO suelen estar dentro de "Pancreas Cancer/" ---
PROJ_ROOT = os.path.dirname(WEIGHTS_DIR.rstrip(os.sep))
PAN_NPZ_ZIP = os.path.join(PROJ_ROOT, "dataset_npz_procesado.zip")
PAN_NPZ_LOCAL = "/content/dataset_npz"


def ensure_pancreas_npz_from_zip():
    if os.path.isdir(PAN_NPZ_LOCAL) and len(glob.glob(os.path.join(PAN_NPZ_LOCAL, "*.npz"))) > 50:
        print(f"[Pancreas NPZ] Ya hay suficientes .npz en {PAN_NPZ_LOCAL}")
        return
    if not os.path.isfile(PAN_NPZ_ZIP):
        print(f"[Pancreas NPZ] No está {PAN_NPZ_ZIP} — súbelo a Drive o coloca .npz en {PAN_NPZ_LOCAL}.")
        return
    import zipfile

    os.makedirs(PAN_NPZ_LOCAL, exist_ok=True)
    print(f"[Pancreas NPZ] Descomprimiendo -> {PAN_NPZ_LOCAL} ...")
    with zipfile.ZipFile(PAN_NPZ_ZIP, "r") as zf:
        zf.extractall(PAN_NPZ_LOCAL)
    n = len(glob.glob(os.path.join(PAN_NPZ_LOCAL, "*.npz")))
    print(f"[Pancreas NPZ] Listo: {n} archivos .npz")


RUN_PAN_NPZ_ZIP = True
if RUN_PAN_NPZ_ZIP:
    ensure_pancreas_npz_from_zip()


# Solo ISIC y Páncreas crudos (nombres del .zip sin extensión; deben coincidir con los de RAW_DIR)
EXTRACT_ZIP_STEMS_ONLY = ("ISIC 2019", "Pancreas Cancer")
RUN_EXTRACT_ZIPS = True
if RUN_EXTRACT_ZIPS:
    extract_datasets_colab(only_zip_stems=EXTRACT_ZIP_STEMS_ONLY)
else:
    print("RUN_EXTRACT_ZIPS=False: se asume que LOCAL_DEST ya contiene los datasets.")


## 3. Rutas ISIC (imágenes + CSV)

Búsqueda bajo `DATASET_ROOTS['ISIC']` (`/content/datasets/ISIC 2019/`), igual que el flujo del router; si falla, se reintenta en todo `LOCAL_DEST`.


In [ ]:
ISIC_ROOT = DATASET_ROOTS["ISIC"][0]


def _find_isic_csv_and_img_dir(search_root):
    """Busca CSV GroundTruth y carpeta con muchas .jpg bajo search_root."""
    csv_path = None
    img_dir = None
    for root, _, files in os.walk(search_root):
        for f in files:
            if f == "ISIC_2019_Training_GroundTruth.csv" or (
                "GroundTruth" in f and f.endswith(".csv") and "ISIC" in f
            ):
                csv_path = os.path.join(root, f)
                break
        if csv_path:
            break
    if csv_path is None:
        for root, _, files in os.walk(search_root):
            for f in files:
                if "GroundTruth" in f and f.endswith(".csv"):
                    csv_path = os.path.join(root, f)
                    break
            if csv_path:
                break
    for root, _, files in os.walk(search_root):
        jpgs = [x for x in files if x.lower().endswith(".jpg")]
        if len(jpgs) > 100:
            img_dir = root
            break
    if img_dir is None:
        cand = os.path.join(search_root, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input")
        img_dir = cand if os.path.isdir(cand) else search_root
    return csv_path, img_dir


CSV_PATH, IMG_DIR = _find_isic_csv_and_img_dir(ISIC_ROOT)
if CSV_PATH is None or not os.path.isfile(CSV_PATH):
    CSV_PATH, IMG_DIR = _find_isic_csv_and_img_dir(LOCAL_DEST)

if CSV_PATH is None or not os.path.isfile(CSV_PATH):
    raise RuntimeError(
        "No se encontró el CSV ISIC. Verifica RAW_DIR, que exista el zip 'ISIC 2019.zip' y que "
        "extract_datasets_colab se haya ejecutado sin errores."
    )

print("ISIC_ROOT:", ISIC_ROOT)
print("CSV_PATH:", CSV_PATH)
print("IMG_DIR :", IMG_DIR)
print("JPGs    :", len(glob.glob(os.path.join(IMG_DIR, "*.jpg"))))


## 4. Experto ISIC — EfficientNet-B3 + Grad-CAM (estilo Osteo)

`GradCAM` / `ScoreCAM` / `LayerCAM`, `ClassifierOutputTarget`, `show_cam_on_image`, capa objetivo = último bloque del backbone (`model.blocks[-1]`).

In [ ]:
from pytorch_grad_cam import GradCAM, ScoreCAM, LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

NUM_CLASSES_VIS = 8
IMG_SIZE = 224
CLASS_NAMES_ISIC = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES_ISIC)}
DATASET_MEAN = np.array([0.485, 0.456, 0.406])
DATASET_STD = np.array([0.229, 0.224, 0.225])

df = pd.read_csv(CSV_PATH)
if "UNK" in df.columns:
    df = df[df["UNK"] != 1.0].reset_index(drop=True)
available_cols = [c for c in CLASS_NAMES_ISIC if c in df.columns]
label_df = df[available_cols]
df["label_name"] = label_df.idxmax(axis=1)
df["label"] = df["label_name"].map(label_to_idx)
img_col = "image" if "image" in df.columns else df.columns[0]
df["path"] = df[img_col].apply(lambda x: os.path.join(IMG_DIR, str(x) + ".jpg"))
df = df[df["path"].apply(os.path.isfile)].reset_index(drop=True)
print("Muestras válidas:", len(df))

# Misma arquitectura y resolución de pesos que el router (05)
model_isic = build_expert(2, pretrained_backbone=False)
ckpt_isic = resolve_checkpoint(_default_checkpoint_candidates(WEIGHTS_DIR)[2])
if not ckpt_isic:
    raise FileNotFoundError(
        "No se encontró checkpoint ISIC: ni .pth ni .pth.zip en WEIGHTS_DIR (exp2_ISIC_EfficientNetB3_best)."
    )
loaded, msg = load_weights(model_isic, ckpt_isic, map_location="cpu", strict=False)
print("Carga ISIC:", loaded, "|", msg, "| ckpt:", ckpt_isic)
if not loaded:
    raise RuntimeError(msg)
model_isic.eval()
model_isic.to(DEVICE)

target_layers_isic = [model_isic.blocks[-1]]
cam_grad = GradCAM(model=model_isic, target_layers=target_layers_isic)
cam_score = ScoreCAM(model=model_isic, target_layers=target_layers_isic)
cam_layer = LayerCAM(model=model_isic, target_layers=target_layers_isic)


def preprocess_isic_path(path):
    from PIL import Image
    im = Image.open(path).convert("RGB")
    im = im.resize((IMG_SIZE, IMG_SIZE))
    arr = np.asarray(im).astype(np.float32) / 255.0
    arr = (arr - DATASET_MEAN) / DATASET_STD
    t = torch.from_numpy(arr).permute(2, 0, 1).float()
    return t


def tensor_to_rgb_vis(tensor_chw, denormalize=True):
    x = tensor_chw.permute(1, 2, 0).detach().cpu().numpy()
    if denormalize:
        x = x * DATASET_STD + DATASET_MEAN
    x = np.clip(x, 0, 1)
    return x.astype(np.float32)


samples_per_class = {c: None for c in range(NUM_CLASSES_VIS)}
for _, row in df.iterrows():
    li = int(row["label"])
    if li >= NUM_CLASSES_VIS:
        continue
    if samples_per_class[li] is None:
        samples_per_class[li] = preprocess_isic_path(row["path"])
    if all(v is not None for v in samples_per_class.values()):
        break

fig, axes = plt.subplots(NUM_CLASSES_VIS, 4, figsize=(14, 2.5 * NUM_CLASSES_VIS))
print("ISIC — Original | Grad-CAM | Mapa calor | Score-CAM / LayerCAM")
print("(MoE define EXPERT_SPECS[2] con num_classes=9; Grad-CAM aquí solo en las 8 clases ISIC.)")

for c in range(NUM_CLASSES_VIS):
    tup = samples_per_class[c]
    if tup is None:
        continue
    img_t = tup.unsqueeze(0).to(DEVICE)
    targets = [ClassifierOutputTarget(c)]
    img_vis = tensor_to_rgb_vis(tup)

    axes[c, 0].imshow(img_vis)
    axes[c, 0].set_title(f"Original — {CLASS_NAMES_ISIC[c]}")
    axes[c, 0].axis("off")

    g = cam_grad(input_tensor=img_t, targets=targets, aug_smooth=True, eigen_smooth=False)[0]
    axes[c, 1].imshow(show_cam_on_image(img_vis, g, use_rgb=True))
    axes[c, 1].set_title("Grad-CAM")
    axes[c, 1].axis("off")
    axes[c, 2].imshow(g, cmap="jet")
    axes[c, 2].set_title("Mapa calor")
    axes[c, 2].axis("off")
    try:
        s = cam_score(input_tensor=img_t, targets=targets)[0]
        axes[c, 3].imshow(show_cam_on_image(img_vis, s, use_rgb=True))
    except Exception:
        l = cam_layer(input_tensor=img_t, targets=targets)[0]
        axes[c, 3].imshow(show_cam_on_image(img_vis, l, use_rgb=True))
    axes[c, 3].set_title("Score / Layer")
    axes[c, 3].axis("off")

plt.suptitle("ISIC 2019 — EfficientNet-B3 — Grad-CAM (metodología Osteo)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Experto Páncreas — R3D-18 + Grad-CAM 3D (última convolución)

Las APIs 2D de `pytorch_grad_cam` no reciben tensores `[B,C,D,H,W]`. El experto 5 del router es **`R3D18Expert`** (1 canal). Aquí: hooks en **`model.backbone[4]`** (layer4), **GAP sobre D×H×W**, mapa 3D y corte axial (overlay).

Los **`.npz`** se buscan en `Pancreas Cancer/` (si existen), en **`/content/dataset_npz`**, en **`.../PROYECTO_MOE_VISION/dataset_npz`** y en subcarpetas. **Si no hay ningún `.npz`**, se toman hasta **`NUM_PAN_SAMPLES`** volúmenes **NIfTI/MHA** bajo `DATASET_ROOTS['Pancreas']` y se aplica **el mismo preproceso que el EDA** (`Pancreas_eda_data_preprocecing.ipynb`): ventana HU **[-1000, 400]**, normalización a [0,1], **`F.interpolate`** trilineal a **64³** (equivalente al array `volume` del NPZ sin guardar archivo).


In [ ]:
# Experto 5: R3D18Expert del 05 (1 canal) — misma clase que load_all_experts_from_drive

model_p = build_expert(5, pretrained_backbone=False)
ckpt_p = resolve_checkpoint(_default_checkpoint_candidates(WEIGHTS_DIR)[5])
if not ckpt_p:
    raise FileNotFoundError("No hay checkpoint páncreas (.pth o .pth.zip) en WEIGHTS_DIR")
loaded_p, msg_p = load_weights(model_p, ckpt_p, map_location="cpu", strict=False)
print("Carga Páncreas:", loaded_p, "|", msg_p, "| ckpt:", ckpt_p)
if not loaded_p:
    raise RuntimeError(msg_p)
model_p.eval()
model_p.to(DEVICE)

PROJ_ROOT = os.path.dirname(WEIGHTS_DIR.rstrip(os.sep))

# --- Constantes alineadas con Pancreas_eda_data_preprocecing.ipynb ---
HU_MIN, HU_MAX = -1000, 400
TARGET_SIZE_PAN = (64, 64, 64)

try:
    import SimpleITK as sitk
except ImportError:
    sitk = None


def collect_pancreas_npz_paths():
    """Mismas fuentes que el EDA/entrenamiento: NPZ no suelen vivir solo en .../Pancreas Cancer/."""
    roots = [
        DATASET_ROOTS["Pancreas"][0],
        "/content/dataset_npz",
        os.path.join(PROJ_ROOT, "dataset_npz"),
    ]
    seen = set()
    out = []
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        for fp in glob.glob(os.path.join(r, "**", "*.npz"), recursive=True):
            if fp not in seen:
                seen.add(fp)
                out.append(fp)
    return sorted(out)


def scan_pancreas_raw_volume_paths(root_dir):
    """NIfTI/MHA bajo la carpeta del dataset (mismas extensiones que el EDA)."""
    out = []
    if not root_dir or not os.path.isdir(root_dir):
        return out
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            low = fn.lower()
            if low.endswith(".nii.gz") or low.endswith(".nii") or low.endswith(".mha"):
                out.append(os.path.join(dirpath, fn))
    return sorted(set(out))


def infer_pancreas_label_from_path(path):
    """Heurística compatible con 05_Router (_infer_pancreas_label_from_path_or_npz)."""
    low = path.lower()
    pos_tokens = ["cancer", "tumor", "positive", "pdac", "malig"]
    neg_tokens = ["normal", "negative", "benign", "non_tumor"]
    if any(t in low for t in pos_tokens):
        return 1
    if any(t in low for t in neg_tokens):
        return 0
    return 0


def preprocess_pancreas_volume_eda_style(file_path):
    """Misma lógica que preprocess_and_save en Pancreas_eda_data_preprocecing.ipynb (sin escribir NPZ)."""
    if sitk is None:
        raise RuntimeError(
            "SimpleITK no está instalado. Ejecuta: pip install SimpleITK"
        )
    img = sitk.ReadImage(file_path)
    arr_raw = sitk.GetArrayFromImage(img).astype(np.float32)
    del img
    np.clip(arr_raw, HU_MIN, HU_MAX, out=arr_raw)
    arr_raw -= HU_MIN
    arr_raw /= (HU_MAX - HU_MIN)
    with torch.no_grad():
        tensor = torch.from_numpy(arr_raw).unsqueeze(0).unsqueeze(0)
        tensor_r = F.interpolate(
            tensor, size=TARGET_SIZE_PAN, mode="trilinear", align_corners=False
        )
        arr_final = tensor_r.squeeze().numpy().copy()
    return arr_final


def load_pancreas_volume_label_dict(path, use_npz):
    if use_npz:
        d = np.load(path, allow_pickle=True)
        vol = d["volume"].astype(np.float32)
        if "label" in d:
            y = int(np.asarray(d["label"]).ravel()[0])
        elif "y" in d:
            y = int(np.asarray(d["y"]).ravel()[0])
        else:
            y = 0
        return vol, y, d
    vol = preprocess_pancreas_volume_eda_style(path)
    y = infer_pancreas_label_from_path(path)
    return vol, y, {"volume": vol}


def volume_to_tensor_zscore(vol):
    t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).float()
    t = (t - t.mean()) / (t.std() + 1e-6)
    return t


def gradcam_r3d_backbone_layer4(model, x, target_class: int):
    acts = []
    grads = []
    layer4 = model.backbone[4]

    def fwd_hook(module, inp, out):
        acts.append(out)

    def bwd_hook(module, grad_in, grad_out):
        if grad_out[0] is not None:
            grads.append(grad_out[0])

    h1 = layer4.register_forward_hook(fwd_hook)
    h2 = layer4.register_full_backward_hook(bwd_hook)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits[0, target_class]
    score.backward()
    h1.remove()
    h2.remove()
    if not acts:
        raise RuntimeError("Sin activaciones layer4")
    A = acts[0]
    G = grads[0] if grads else torch.autograd.grad(score, A)[0]
    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = (w * A).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = cam[0, 0]
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy(), logits.detach().cpu()


npz_list = collect_pancreas_npz_paths()
print(
    "Buscando .npz en:",
    DATASET_ROOTS["Pancreas"][0],
    ", /content/dataset_npz,",
    os.path.join(PROJ_ROOT, "dataset_npz"),
)
print("Total NPZ encontrados:", len(npz_list))

NUM_PAN_SAMPLES = 4
random.seed(42)

if len(npz_list) >= 1:
    use_npz = True
    pool = npz_list
    print("Modo: volúmenes desde archivos .npz")
else:
    use_npz = False
    pan_root = DATASET_ROOTS["Pancreas"][0]
    pool = scan_pancreas_raw_volume_paths(pan_root)
    print(
        "[Pancreas] Sin .npz: buscando .nii / .nii.gz / .mha en:",
        pan_root,
        "-> encontrados:",
        len(pool),
    )
    if len(pool) < 1:
        raise RuntimeError(
            "No hay archivos .npz ni volúmenes NIfTI/MHA bajo Pancreas. Opciones: "
            "(1) Descomprime dataset_npz_procesado.zip a /content/dataset_npz, "
            "(2) Copia .npz ahí o a "
            f"{os.path.join(PROJ_ROOT, 'dataset_npz')}, "
            "(3) Coloca al menos un .nii/.mha bajo "
            f"{pan_root}, (4) Genera NPZ con notebooks/Expertos y doc/Pancreas_eda_*.ipynb."
        )
    if sitk is None:
        raise RuntimeError(
            "Sin NPZ hace falta SimpleITK para leer NIfTI/MHA. Ejecuta la celda 0: pip install SimpleITK"
        )

n_pick = min(NUM_PAN_SAMPLES, len(pool))
sample_paths = random.sample(pool, n_pick)

fig, axes = plt.subplots(n_pick, 3, figsize=(10, 3.5 * n_pick))
if n_pick == 1:
    axes = axes.reshape(1, -1)

for row, pth in enumerate(sample_paths):
    vol, y, d_raw = load_pancreas_volume_label_dict(pth, use_npz)
    x = volume_to_tensor_zscore(vol).to(DEVICE)
    x.requires_grad_(True)
    cam3, logits = gradcam_r3d_backbone_layer4(model_p, x, target_class=y)
    pred = int(logits.argmax(dim=-1).item())
    D = cam3.shape[0]
    sl = D // 2
    vol_raw = d_raw["volume"].astype(np.float32)
    sl_img = vol_raw[sl]
    sl_img = (sl_img - sl_img.min()) / (sl_img.max() - sl_img.min() + 1e-8)
    heat = cam3[sl]
    src = os.path.basename(pth)
    title0 = f"{src[:28]}… | y={y}" if len(src) > 30 else f"{src} | y={y}"
    axes[row, 0].imshow(sl_img, cmap="gray")
    axes[row, 0].set_title(title0)
    axes[row, 0].axis("off")
    axes[row, 1].imshow(heat, cmap="jet")
    axes[row, 1].set_title("Grad-CAM 3D (corte)")
    axes[row, 1].axis("off")
    axes[row, 2].imshow(sl_img, cmap="gray")
    axes[row, 2].imshow(heat, cmap="jet", alpha=0.45)
    axes[row, 2].set_title(f"Overlay | pred={pred}")
    axes[row, 2].axis("off")

plt.suptitle("Páncreas — R3D18Expert (05) — Grad-CAM 3D (backbone[4])", fontsize=12)
plt.tight_layout()
plt.show()


## Nota: inferencia solo (sin Grad-CAM)

Para **solo logits** en CPU/GPU: `model.eval()`, `with torch.no_grad(): y = model(x)`. Grad-CAM **necesita** gradientes en la capa objetivo; en CPU es válido pero más lento.